# Building a Multi-Turn Request with the Messages API

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

The Anthropic SDK and `python-dotenv` are all this notebook needs. We install with **uv** because uv-managed venvs ship without `pip`, so `%pip install` fails inside this kernel.

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

`load_dotenv()` reads a local `.env` file and sets `ANTHROPIC_API_KEY` in the environment. The SDK client reads that key automatically, so **the key never appears in code**.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

`Anthropic()` builds the client with the key it found in the environment. We pin **`model`** to `claude-haiku-4-5`, the course default. Every request in this notebook reuses this one client.

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

The Messages API is **stateless**: it holds no memory between calls. A conversation is just a `messages` list you carry forward, alternating **`user`** and **`assistant`** roles.

| Helper | Role appended | Purpose |
| --- | --- | --- |
| `add_user_message` | `user` | records your turn |
| `add_assistant_message` | `assistant` | records Claude's reply into history |
| `chat` | none | sends the whole list, returns the text |

> Every turn you want Claude to remember must be appended back into `messages` before the next `chat` call.

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

### 5. Build the conversation turn by turn

We start with an empty list, add a **user** question, get the reply, append that reply as an **assistant** turn, then add a follow-up. The follow-up carries the **full history**, so Claude answers "Write another sentence" with the earlier topic in context.

In [5]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
add_assistant_message(messages, final_answer)

### 6. Parse the response

Inspecting `messages` shows the accumulated history: alternating **`user`** and **`assistant`** entries. This list is the conversation - persist it, and you can resume the thread on any later request.

In [6]:
messages

[{'role': 'user', 'content': 'Define quantum computing in one sentence'},
 {'role': 'assistant',
  'content': 'Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, potentially solving certain complex problems exponentially faster.'},
 {'role': 'user', 'content': 'Write another sentence'},
 {'role': 'assistant',
  'content': 'Unlike classical computers that process information as binary bits (0 or 1), quantum computers use quantum bits or "qubits" that can exist in multiple states simultaneously, enabling them to explore many possible solutions in parallel.'}]